Pre-process CMP, admissions, lab results, and diagnosis data

In [1]:
from pathlib import Path
import gc
import pandas as pd
import numpy as np

# Display options for notebook inspection
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# File paths
RAW = Path("data/raw")
PROCESSED = Path("data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

1.) Load tables from source data

In [2]:
patients = pd.read_csv(
    RAW / "patients.csv.gz",
    usecols=["subject_id", "gender", "anchor_age", "anchor_year"],
    dtype={
        "subject_id": "int32",
        "gender": "category",
        "anchor_age": "float32",
        "anchor_year": "float32"
    }
)

admissions = pd.read_csv(
    RAW / "admissions.csv.gz",
    usecols=["subject_id", "hadm_id", "admittime", "admission_type", "race"],
    dtype={
        "subject_id": "int32",
        "hadm_id": "int32",
        "admission_type": "category",
        "race": "category"
    },
    parse_dates=["admittime"]
)

d_labitems = pd.read_csv(
    RAW / "d_labitems.csv.gz",
    usecols=["itemid", "label", "fluid", "category"],
    dtype={
        "itemid": "int32",
        "label": "string",
        "fluid": "category",
        "category": "category"
    }
)

diagnoses = pd.read_csv(
    RAW / "diagnoses_icd.csv.gz",
    usecols=["subject_id", "hadm_id", "icd_code", "icd_version"],
    dtype={
        "subject_id": "int32",
        "hadm_id": "int32",
        "icd_code": "string",
        "icd_version": "int8"
    }
)

omr = pd.read_csv(
    RAW / "omr.csv.gz",
    parse_dates=["chartdate"]
)

print("patients:", patients.shape)
print("admissions:", admissions.shape)
print("d_labitems:", d_labitems.shape)
print("diagnoses:", diagnoses.shape)
print("omr:", omr.shape)

patients: (364627, 4)
admissions: (546028, 5)
d_labitems: (1650, 4)
diagnoses: (6364488, 4)
omr: (7753027, 5)


2.) Define CMP components

In [3]:
CMP_LABEL_MAP = {
    "Alanine Aminotransferase (ALT)": "alt",
    "Albumin": "albumin",
    "Alkaline Phosphatase": "alp",
    "Asparate Aminotransferase (AST)": "ast",
    "Bicarbonate": "bicarbonate",
    "Bilirubin, Total": "total_bilirubin",
    "Calcium, Total": "total_calcium",
    "Chloride": "chloride",
    "Creatinine": "creatinine",
    "Glucose": "glucose",
    "Potassium": "potassium",
    "Protein, Total": "total_protein",
    "Sodium": "sodium",
    "Urea Nitrogen": "bun"
}

cmp_labitems = d_labitems[d_labitems["label"].isin(CMP_LABEL_MAP)].copy()
cmp_labitems["component"] = cmp_labitems["label"].map(CMP_LABEL_MAP)

cmp_itemids = set(cmp_labitems["itemid"])

3.) Filter lab events to CMP related rows

In [4]:
labevents_cols = [
    "subject_id",
    "hadm_id",
    "itemid",
    "charttime",
    "valuenum",
    "flag"
]

chunk_iter = pd.read_csv(
    RAW / "labevents.csv.gz",
    usecols=labevents_cols,
    dtype={
        "subject_id": "int32",
        "hadm_id": "float64",
        "itemid": "int32",
        "valuenum": "float32",
        "flag": "string"
    },
    parse_dates=["charttime"],
    chunksize=1_000_000
)

cmp_labs_file = PROCESSED / "cmp_labs_filtered.csv"
first_write = True

for i, chunk in enumerate(chunk_iter, start=1):
    chunk = chunk[chunk["itemid"].isin(cmp_itemids)].copy()
    chunk = chunk.dropna(subset=["hadm_id", "charttime", "valuenum"])

    if not chunk.empty:
        chunk["hadm_id"] = chunk["hadm_id"].astype("int32")

        chunk.to_csv(
            cmp_labs_file,
            mode="w" if first_write else "a",
            header=first_write,
            index=False
        )
        first_write = False

    print(f"Processed chunk {i:,}")

    del chunk
    gc.collect()

Processed chunk 1
Processed chunk 2
Processed chunk 3
Processed chunk 4
Processed chunk 5
Processed chunk 6
Processed chunk 7
Processed chunk 8
Processed chunk 9
Processed chunk 10
Processed chunk 11
Processed chunk 12
Processed chunk 13
Processed chunk 14
Processed chunk 15
Processed chunk 16
Processed chunk 17
Processed chunk 18
Processed chunk 19
Processed chunk 20
Processed chunk 21
Processed chunk 22
Processed chunk 23
Processed chunk 24
Processed chunk 25
Processed chunk 26
Processed chunk 27
Processed chunk 28
Processed chunk 29
Processed chunk 30
Processed chunk 31
Processed chunk 32
Processed chunk 33
Processed chunk 34
Processed chunk 35
Processed chunk 36
Processed chunk 37
Processed chunk 38
Processed chunk 39
Processed chunk 40
Processed chunk 41
Processed chunk 42
Processed chunk 43
Processed chunk 44
Processed chunk 45
Processed chunk 46
Processed chunk 47
Processed chunk 48
Processed chunk 49
Processed chunk 50
Processed chunk 51
Processed chunk 52
Processed chunk 53
Pr

4.) Construct CMP event rows

In [5]:
cmp_labs = pd.read_csv(
    cmp_labs_file,
    dtype={
        "subject_id": "int32",
        "hadm_id": "int32",
        "itemid": "int32",
        "valuenum": "float32",
        "flag": "string"
    },
    parse_dates=["charttime"]
)

cmp_labs = cmp_labs.merge(
    cmp_labitems[["itemid", "component"]],
    on="itemid",
    how="left"
)

# Mark abnormal CMP components using the provided lab flag values
cmp_labs["component_abnormal"] = (
    cmp_labs["flag"]
    .fillna("")
    .str.upper()
    .isin(["ABNORMAL", "HIGH", "LOW"])
    .astype("int8")
)

# Pivot numeric component values to wide format
cmp_wide_values = cmp_labs.pivot_table(
    index=["subject_id", "hadm_id", "charttime"],
    columns="component",
    values="valuenum",
    aggfunc="first"
).reset_index()

# Pivot abnormal flags to wide format
cmp_wide_flags = cmp_labs.pivot_table(
    index=["subject_id", "hadm_id", "charttime"],
    columns="component",
    values="component_abnormal",
    aggfunc="max"
).reset_index()

flag_cols = {
    col: f"{col}_abnormal"
    for col in cmp_wide_flags.columns
    if col not in ["subject_id", "hadm_id", "charttime"]
}
cmp_wide_flags = cmp_wide_flags.rename(columns=flag_cols)

# Merge values and abnormal flags into one CMP event table
cmp_events = cmp_wide_values.merge(
    cmp_wide_flags,
    on=["subject_id", "hadm_id", "charttime"],
    how="left"
)

# Define target: whether any CMP component in the event is abnormal
abnormal_cols = [c for c in cmp_events.columns if c.endswith("_abnormal")]
cmp_events["target_abnormal"] = (
    cmp_events[abnormal_cols]
    .fillna(0)
    .max(axis=1)
    .astype("int8")
)

# cmp_events.to_csv(PROCESSED / "cmp_events.csv", index=False)

print("cmp_events:", cmp_events.shape)
print(cmp_events["target_abnormal"].value_counts(normalize=True))

cmp_events: (2988151, 32)
target_abnormal
1    0.888252
0    0.111748
Name: proportion, dtype: float64


5.) Add prior component values, deltas, time, and abnormal count

In [6]:
cmp_events = cmp_events.sort_values(["subject_id", "charttime"]).copy()

components = [c for c in CMP_LABEL_MAP.values() if c in cmp_events.columns]

for comp in components:
    cmp_events[f"prev_{comp}"] = cmp_events.groupby("subject_id")[comp].shift(1)
    cmp_events[f"delta_{comp}"] = (
        cmp_events[comp] - cmp_events[f"prev_{comp}"]
    ).astype("float32")

cmp_events["prev_cmp_time"] = cmp_events.groupby("subject_id")["charttime"].shift(1)

cmp_events["hours_since_last_cmp"] = (
    (cmp_events["charttime"] - cmp_events["prev_cmp_time"]).dt.total_seconds() / 3600
).astype("float32")

cmp_events["prior_abnormal_cmp_count"] = (
    cmp_events.groupby("subject_id")["target_abnormal"]
    .cumsum()
    .shift(1)
    .fillna(0)
    .astype("int32")
)

cmp_events["abnormal_last_5_events"] = (
    cmp_events.groupby("subject_id")["target_abnormal"]
    .transform(lambda s: s.shift(1).rolling(5, min_periods=1).sum())
    .fillna(0)
    .astype("int")
)

# cmp_events.to_csv(PROCESSED / "cmp_events_with_prior.csv", index=False)

print("cmp_events_with_prior:", cmp_events.shape)

cmp_events_with_prior: (2988151, 64)


6.) Add demographics and admissions information

In [7]:
cmp_events = cmp_events.merge(
    admissions,
    on=["subject_id", "hadm_id"],
    how="left"
)

cmp_events = cmp_events.merge(
    patients,
    on="subject_id",
    how="left"
)

cmp_events["age"] = cmp_events["anchor_age"].astype("float32")
cmp_events["sex"] = cmp_events["gender"].astype("category")

# Hospital stay at the CMP event time
cmp_events["length_of_stay_hours"] = (
    (cmp_events["charttime"] - cmp_events["admittime"]).dt.total_seconds() / 3600
).astype("float32")

# Exclude rows where CMP time is before admission time
cmp_events = cmp_events[cmp_events["length_of_stay_hours"] >= 0].copy()

# cmp_events.to_csv(PROCESSED / "cmp_events_with_demo.csv", index=False)

print("cmp_events_with_demo:", cmp_events.shape)

cmp_events_with_demo: (2914207, 73)


7.) Add vitals

In [8]:
wanted_results = ["Blood Pressure", "BMI (kg/m2)"]
vitals = omr[omr["result_name"].isin(wanted_results)].copy()

# Ensure subject_id is numeric
vitals["subject_id"] = pd.to_numeric(vitals["subject_id"], errors="coerce")
vitals = vitals.dropna(subset=["subject_id", "chartdate"]).copy()
vitals["subject_id"] = vitals["subject_id"].astype("int32")

# Process numeric vitals
simple_vitals = vitals[vitals["result_name"] == "BMI (kg/m2)"].copy()
simple_vitals["vital"] = "bmi"
simple_vitals["result_value"] = pd.to_numeric(simple_vitals["result_value"], errors="coerce")
simple_vitals = simple_vitals.dropna(subset=["result_value"])

simple_wide = simple_vitals.pivot_table(
    index=["subject_id", "chartdate"],
    columns="vital",
    values="result_value",
    aggfunc="last"
).reset_index()

# Process blood pressure into systolic and diastolic values
bp = vitals[vitals["result_name"] == "Blood Pressure"].copy()
bp_parts = bp["result_value"].astype(str).str.extract(r"^\s*(\d{2,3})\s*/\s*(\d{2,3})\s*$")
bp["sbp"] = pd.to_numeric(bp_parts[0], errors="coerce")
bp["dbp"] = pd.to_numeric(bp_parts[1], errors="coerce")
bp = bp.dropna(subset=["sbp", "dbp"])

bp_wide = (
    bp[["subject_id", "chartdate", "sbp", "dbp"]]
    .sort_values(["subject_id", "chartdate"])
    .groupby(["subject_id", "chartdate"], as_index=False)
    .last()
)

# Combine BMI and blood pressure into one feature table
vitals_wide = pd.merge(
    simple_wide,
    bp_wide,
    on=["subject_id", "chartdate"],
    how="outer"
)

In [9]:
# Prepare both tables
cmp_events = cmp_events.dropna(subset=["subject_id", "charttime"]).copy()
vitals_wide = vitals_wide.dropna(subset=["subject_id", "chartdate"]).copy()

cmp_events = cmp_events.sort_values(["charttime", "subject_id"]).reset_index(drop=True)
vitals_wide = vitals_wide.sort_values(["chartdate", "subject_id"]).reset_index(drop=True)

final_df = pd.merge_asof(
    cmp_events,
    vitals_wide,
    left_on="charttime",
    right_on="chartdate",
    by="subject_id",
    direction="backward"
)

print("final_df after vitals merge:", final_df.shape)

final_df after vitals merge: (2914207, 77)


8.) Add diagnoses

In [10]:
diagnoses["icd_code"] = diagnoses["icd_code"].str.upper().str.strip()
diagnoses = diagnoses.dropna(subset=["hadm_id", "icd_code"]).copy()

# Keep ICD version in the grouped label to avoid mixing ICD-9 and ICD-10 prefixes
diagnoses["icd3_group"] = (
    "ICD" + diagnoses["icd_version"].astype(str) + "_" + diagnoses["icd_code"].str[:3]
)

# Keep the most common grouped diagnoses and collapse the rest into OTHER
top_n = 10
top_diag_groups = diagnoses["icd3_group"].value_counts().head(top_n).index.tolist()

diagnoses["diag_group_final"] = diagnoses["icd3_group"].where(
    diagnoses["icd3_group"].isin(top_diag_groups),
    other="OTHER"
)

diag_flags = pd.crosstab(
    diagnoses["hadm_id"],
    diagnoses["diag_group_final"]
).reset_index()

diag_flags.columns = [
    "hadm_id" if col == "hadm_id" else f"dx_{str(col).lower()}"
    for col in diag_flags.columns
]

final_df = final_df.merge(diag_flags, on="hadm_id", how="left")

diag_cols = [c for c in final_df.columns if c.startswith("dx_")]
final_df[diag_cols] = final_df[diag_cols].fillna(0).astype("int8")

print("Diagnosis feature columns:", len(diag_cols))

Diagnosis feature columns: 11


9.) Final modeling columns, save dataset

In [11]:
keep_cols = [
    "subject_id",
    "hadm_id",
    "charttime",
    "age",
    "sex",
    "race",
    "admission_type",
    "length_of_stay_hours",
    "hours_since_last_cmp",
    "prior_abnormal_cmp_count",
    "abnormal_last_5_events",
    "bmi",
    "sbp",
    "dbp",
    "target_abnormal"
]

prior_cols = [c for c in final_df.columns if c.startswith("prev_")]
delta_cols = [c for c in final_df.columns if c.startswith("delta_")]
diag_cols = [c for c in final_df.columns if c.startswith("dx_")]

keep_cols.extend(prior_cols)
keep_cols.extend(delta_cols)
keep_cols.extend(diag_cols)

keep_cols = [c for c in keep_cols if c in final_df.columns]

model_df = final_df[keep_cols].copy()
model_df = model_df.dropna(subset=["target_abnormal"]).copy()

model_df.to_csv(PROCESSED / "final_model_df.csv", index=False)

print("final_model_df:", model_df.shape)
display(model_df.head())

final_model_df: (2914207, 55)


,subject_id,hadm_id,charttime,age,sex,race,admission_type,length_of_stay_hours,hours_since_last_cmp,prior_abnormal_cmp_count,abnormal_last_5_events,bmi,sbp,dbp,target_abnormal,prev_alt,prev_albumin,prev_alp,prev_ast,prev_bicarbonate,prev_total_bilirubin,prev_total_calcium,prev_chloride,prev_creatinine,prev_glucose,prev_potassium,prev_total_protein,prev_sodium,prev_bun,prev_cmp_time,delta_alt,delta_albumin,delta_alp,delta_ast,delta_bicarbonate,delta_total_bilirubin,delta_total_calcium,delta_chloride,delta_creatinine,delta_glucose,delta_potassium,delta_total_protein,delta_sodium,delta_bun,dx_icd10_e11,dx_icd10_e78,dx_icd10_i10,dx_icd10_i25,dx_icd10_z79,dx_icd10_z87,dx_icd9_250,dx_icd9_272,dx_icd9_401,dx_icd9_v58,dx_other
0,16904137,21081215,2105-10-05 01:41:00,54.0,M,OTHER,URGENT,8.250000,NaN,1,0,NaN,122.0,78.0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,1,0,0,0,7
1,16904137,21081215,2105-10-05 07:47:00,54.0,M,OTHER,URGENT,14.350000,6.100000,1,1,NaN,122.0,78.0,1,NaN,NaN,NaN,NaN,23.0,NaN,7.3,109.0,1.0,104.0,3.6,NaN,137.0,22.0,2105-10-05 01:41:00,NaN,NaN,NaN,NaN,-2.0,NaN,-0.2,1.0,0.0,113.0,-0.1,NaN,1.0,-2.0,0,0,0,0,0,0,1,0,0,0,7
2,16904137,21081215,2105-10-05 11:44:00,54.0,M,OTHER,URGENT,18.299999,3.950000,2,2,NaN,122.0,78.0,1,14.0,2.1,25.0,19.0,21.0,0.2,7.1,110.0,1.0,217.0,3.5,NaN,138.0,20.0,2105-10-05 07:47:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,25.0,NaN,NaN,NaN,NaN,0,0,0,0,0,0,1,0,0,0,7
3,16904137,21081215,2105-10-05 12:41:00,54.0,M,OTHER,URGENT,19.250000,0.950000,3,3,NaN,122.0,78.0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,242.0,NaN,NaN,NaN,NaN,2105-10-05 11:44:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-6.0,NaN,NaN,NaN,NaN,0,0,0,0,0,0,1,0,0,0,7
4,16904137,21081215,2105-10-05 14:01:00,54.0,M,OTHER,URGENT,20.583334,1.333333,4,4,NaN,122.0,78.0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,236.0,NaN,NaN,NaN,NaN,2105-10-05 12:41:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.0,NaN,NaN,NaN,NaN,0,0,0,0,0,0,1,0,0,0,7
